# 05 · Enhancement 1: DA3 Self-Audit, original DA3 vs DA3 + Self-Audit
Training-free enhancement on the **released** weights (setting **[official]**); the network is not
changed. DA3 predicts each camera twice (camera head, ray map), over-parameterises it (6-D ray map)
and can take its own cameras back through E_c: the Self-Audit turns these redundancies into
reliability information. Plan and novelty: `docs/da3_enhancement_report.md` §4.2; how to run and
how to save the results: `docs/ENHANCEMENTS_RUNBOOK.md`.

| | Original DA3 | DA3 + Self-Audit |
|---|---|---|
| **A. Failure detection** (per view) | its own depth confidence | audit model on the redundancy signals (fitted leave-one-dataset-out) |
| **B. Pose source** (paper protocol, Sec. 6) | camera-head poses (API default); ray-map poses (official option) | per-view choice / fusion of head and ray poses (+ oracle as upper bound) |
| **C. Depth uncertainty** (HiRoom, 7Scenes) | confidence calibrated with GT on *other* datasets | per-scene triple collocation, **no GT** |

Settings: **Accelerator = GPU (T4 / P100)**, **Internet = on**. One GPU pass per model (baseline
benchmark run + Self-Audit dump with one extra E_c pass per scene); everything after that is CPU
work (signals, harness replays with the same depth and different cameras, TSDF scoring).
`PHASE = "analyse"` with `RESUME_FROM` re-runs only the analysis (a CPU session is enough).
Outputs (in `/kaggle/working/`): `self_audit/<tag>/` (summary.md, figures, per_view.csv),
`self_audit/results/` (standard result records: **copy them into the repo under
`experiments/results/` and commit**, so notebook 09 can compare all enhancements),
`self_audit_results.zip`.

In [ ]:
# --- repo setup (identical in every notebook) ---------------------------------------------
REPO_URL = "https://github.com/davideCabitz/DA3_LAB_Advanced_CV.git"   # push your branch first
BRANCH = "main"
import os, subprocess, sys
os.chdir("/kaggle/working")
url = REPO_URL
try:  # private repo: add a Kaggle secret named GITHUB_TOKEN (Add-ons > Secrets)
    from kaggle_secrets import UserSecretsClient
    tok = UserSecretsClient().get_secret("GITHUB_TOKEN")
    url = REPO_URL.replace("https://", f"https://{tok}@")
except Exception:
    pass
if not os.path.exists("DA3_LAB_Advanced_CV"):
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, "DA3_LAB_Advanced_CV"], check=True)
else:
    subprocess.run(["git", "-C", "DA3_LAB_Advanced_CV", "pull", "--ff-only"], check=True)
os.chdir("/kaggle/working/DA3_LAB_Advanced_CV")
!pip install -q -r requirements.txt && pip install -q -e . --no-deps
sys.path.insert(0, os.path.abspath("src"))   # the editable install is only seen by new processes, not this kernel
!git log -1 --format="commit %H  %cd"

In [ ]:
# --- official DA3 repo (reference / parity oracle / benchmark harness), pinned commit ----------
import os, subprocess
if not os.path.exists("/kaggle/working/Depth-Anything-3"):
    subprocess.run(["git", "clone", "https://github.com/ByteDance-Seed/Depth-Anything-3.git",
                    "/kaggle/working/Depth-Anything-3"], check=True)
subprocess.run(["git", "-C", "/kaggle/working/Depth-Anything-3", "checkout", "3d835ec1a5802d64a8b8b15f817a1ab54809bfe4"], check=True)
OFFICIAL_SRC = "/kaggle/working/Depth-Anything-3/src"
!pip install -q -r requirements-eval.txt
# Open3D self-test: does TSDF fusion (depth maps -> mesh) work in this environment?
!python scripts/check_open3d.py --out /kaggle/working/open3d_check.json

In [ ]:
import json
chk = json.load(open("/kaggle/working/open3d_check.json"))
RECON = bool(chk["legacy_tsdf_ok"])   # reconstruction scores (F1 / Chamfer) need a working Open3D TSDF
if RECON:
    print("Open3D TSDF OK:", chk["open3d"], "-> reconstruction scores (F1 / Chamfer) included")
else:
    print("Open3D", chk["open3d"], "on Python", chk["python"], "has a broken TSDF fusion (known Open3D 0.20 bug, "
          "github.com/isl-org/Open3D/issues/7592; the working 0.19.0 only exists up to Python 3.12).")
    print("This run SKIPS the reconstruction scores (F1 / Chamfer). Pose accuracy (AUC), failure detection "
          "and depth uncertainty are not affected.")
MODES = "pose recon_unposed" if RECON else "pose"
RECON_VARIANTS = "" if RECON else "--recon-variants"   # an empty list = pose-only harness replays

### Configuration

In [ ]:
MODELS = ["da3-small"]                      # full runs: ONE model per session ("da3-small", then "da3-base")
DATASETS = ["hiroom", "7scenes", "dtu64"]   # add "eth3d", "scannetpp" in a later session (15 / 11 GB)
MAX_SCENES = 2        # 2 = quick end-to-end check (labelled "limited"); 0 = all scenes (the real numbers)
PHASE = "all"         # "all" | "infer" (GPU: baseline run + dump) | "analyse" (CPU is enough)
RESUME_FROM = ""      # output root of an earlier "infer" run of this notebook, e.g. "/kaggle/input/<name>"
WORK = "/tmp/eval_self_audit"               # harness exports + audit dumps (large, not saved)
# split mode only: for PHASE = "infer" use WORK = "/kaggle/working/eval_self_audit" so the dump is saved
OUT = "/kaggle/working/self_audit"          # figures, CSV, summaries
RESULTS = f"{OUT}/results"                  # standard result records (commit these)
BENCH = "/tmp/da3bench"
DS = " ".join(DATASETS)
def tag_for(m):
    return f"sa_{m}_{'-'.join(DATASETS)}" + (f"_limited{MAX_SCENES}" if MAX_SCENES else "")
os.makedirs(RESULTS, exist_ok=True)
if RESUME_FROM:
    !mkdir -p {WORK} && cp -rn {RESUME_FROM}/eval_self_audit/* {WORK}/
    !cp -rn {RESUME_FROM}/self_audit/results/* {RESULTS}/ 2>/dev/null; ls {RESULTS}
print({m: tag_for(m) for m in MODELS})

### Data: DA3-BENCH subsets (HF `depth-anything/DA3-BENCH`)

In [ ]:
from huggingface_hub import hf_hub_download
import zipfile
os.makedirs(BENCH, exist_ok=True)
for d in DATASETS:
    if not os.path.exists(f"{BENCH}/{d}"):
        z = hf_hub_download("depth-anything/DA3-BENCH", f"{d}.zip", repo_type="dataset", local_dir="/tmp/zips")
        zipfile.ZipFile(z).extractall(BENCH); os.remove(z)
!du -sh {BENCH}/*

### 1 · Original DA3 on the paper protocol + Self-Audit dump (GPU)
Released harness, camera-head poses (API default). `--dump-audit` also saves, next to each scene's
export, the ray-map poses (official RANSAC routine and our DLT), a pooled ray map and the E_c
re-pass (DA3 conditioned on its own cameras). The exported depth / poses are unchanged.

In [ ]:
if PHASE in ("all", "infer"):
    for m in MODELS:
        t = tag_for(m)
        !python scripts/eval_bench.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK} \
            --weights official:{m} --datasets {DS} --modes {MODES} --max-scenes {MAX_SCENES} \
            --tag {t} --enhancement baseline --variant head --dump-audit --results-root {RESULTS}
!du -sh {WORK}

### 2 · Self-Audit analysis (CPU)
Signals → per-view GT errors → audit model (leave-one-dataset-out) → A and C metrics, then the
pose-source variants are written with the **same depth** and scored by the harness (`--eval-only`).

In [ ]:
if PHASE in ("all", "analyse"):
    for m in MODELS:
        t = tag_for(m)
        !python scripts/self_audit_eval.py --official-src {OFFICIAL_SRC} --bench-root {BENCH} --work-root {WORK} \
            --baseline-tag {t} --weights official:{m} --datasets {DS} --max-scenes {MAX_SCENES} \
            --results-root {RESULTS} --out-dir {OUT}/{t} {RECON_VARIANTS}

### 3 · Results: original DA3 vs DA3 + Self-Audit

In [ ]:
from IPython.display import Image, Markdown, display
for m in MODELS:
    p = f"{OUT}/{tag_for(m)}/summary.md"
    display(Markdown(open(p).read()) if os.path.exists(p) else Markdown(f"_missing: {p}_"))

In [ ]:
import matplotlib.pyplot as plt
from da3.compare import load_records
from da3.compare.plots import sec6_delta_bars
recs = load_records(RESULTS)
fig, axes = plt.subplots(1, 3, figsize=(18, 4))
for ax, lab in zip(axes, ["AUC@3", "AUC@30", "F1 w/o p."]):
    sec6_delta_bars(recs, lab, ax=ax)
plt.tight_layout(); plt.show()

In [ ]:
for m in MODELS:
    for f in ["risk_coverage.png", "depth_reliability.png", "tc_vs_gt_sigma.png", "audit_coefficients.png", "audit_examples.png"]:
        p = f"{OUT}/{tag_for(m)}/{f}"
        if os.path.exists(p):
            print(m, "·", f); display(Image(p, width=1100))

### 4 · Go/no-go gates (report §4.2)

In [ ]:
for r in load_records(RESULTS):
    if r["enhancement"] == "self_audit" and "gates" in r["extra"]:
        print(f"== {r['weights']} ({r['tag']})")
        for name, g in r["extra"]["gates"].items():
            print(f"  {g['result']:5s} {name}: {g['value']}")

### 5 · Save: download `self_audit_results.zip`, copy `self_audit/results/*` into the repo's `experiments/results/`, commit

In [ ]:
!cd /kaggle/working && rm -f self_audit_results.zip && zip -qr self_audit_results.zip self_audit && ls -la self_audit_results.zip
!find {RESULTS} -name "*.json" | sort